In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("All imports successful!")

C:\Users\Muzammil Ahmed\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful!


In [2]:
from pathlib import Path

# Project and knowledge-base paths
PROJECT_DIR = Path.cwd()
KB_DIR = PROJECT_DIR / "knowledge_base"

# Find all .txt files
documents = list(KB_DIR.rglob("*.txt"))

print(f"Knowledge-base documents found: {len(documents)}")

for doc in documents:
    print(doc.relative_to(KB_DIR))

Knowledge-base documents found: 12
faq\retailer_faq.txt
policies\delivery_policy.txt
policies\installation_policy.txt
policies\return_refund_policy.txt
policies\warranty_policy.txt
products\air_conditioners.txt
products\microwave_ovens.txt
products\refrigerators.txt
products\washing_machines.txt
support\air_conditioner_troubleshooting.txt
support\refrigerator_troubleshooting.txt
support\washing_machine_troubleshooting.txt


In [3]:
documents_data = []

for file_path in documents:
    text = file_path.read_text(encoding="utf-8")

    # Get category from folder structure
    relative_path = file_path.relative_to(KB_DIR)
    category = relative_path.parts[0]

    documents_data.append({
        "source": file_path.name,
        "path": str(relative_path),
        "category": category,
        "content": text.strip()
    })

# Convert to DataFrame for easy inspection
df_docs = pd.DataFrame(documents_data)

print(f"Documents loaded: {len(df_docs)}")
print()
print(df_docs[["source", "category"]].to_string(index=False))

Documents loaded: 12

                             source category
                   retailer_faq.txt      faq
                delivery_policy.txt policies
            installation_policy.txt policies
           return_refund_policy.txt policies
                warranty_policy.txt policies
               air_conditioners.txt products
                microwave_ovens.txt products
                  refrigerators.txt products
               washing_machines.txt products
air_conditioner_troubleshooting.txt  support
   refrigerator_troubleshooting.txt  support
washing_machine_troubleshooting.txt  support


In [4]:
import re

def clean_text(text):
    """Clean unnecessary spaces while preserving the actual information."""
    text = text.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def create_chunks(text, chunk_size=800, overlap=100):
    """Split text into overlapping character-based chunks."""
    text = clean_text(text)

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks


# Create chunks for every document
chunks_data = []

for _, row in df_docs.iterrows():
    chunks = create_chunks(row["content"])

    for i, chunk in enumerate(chunks):
        chunks_data.append({
            "source": row["source"],
            "path": row["path"],
            "category": row["category"],
            "chunk_id": i,
            "content": chunk
        })

df_chunks = pd.DataFrame(chunks_data)

print(f"Documents: {len(df_docs)}")
print(f"Total chunks: {len(df_chunks)}")
print()
print(df_chunks[["source", "chunk_id"]].head(10).to_string(index=False))

Documents: 12
Total chunks: 21

                  source  chunk_id
        retailer_faq.txt         0
        retailer_faq.txt         1
     delivery_policy.txt         0
 installation_policy.txt         0
 installation_policy.txt         1
return_refund_policy.txt         0
return_refund_policy.txt         1
     warranty_policy.txt         0
     warranty_policy.txt         1
    air_conditioners.txt         0


In [5]:
# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert every chunk into an embedding vector
embeddings = embedding_model.encode(
    df_chunks["content"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

# Convert to NumPy format required by FAISS
embeddings = np.array(embeddings, dtype="float32")

print("Embeddings created successfully!")
print("Number of embeddings:", embeddings.shape[0])
print("Vector dimensions:", embeddings.shape[1])

C:\Users\Muzammil Ahmed\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Muzammil Ahmed\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Muzammil Ahmed\AppData\Local\Python\pythoncore-3

Embeddings created successfully!
Number of embeddings: 21
Vector dimensions: 384


In [6]:
# Create a FAISS index for cosine similarity
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

# Add our document embeddings to the index
index.add(embeddings)

print("FAISS index created successfully!")
print("Vectors stored:", index.ntotal)

FAISS index created successfully!
Vectors stored: 21


In [7]:
def retrieve_chunks(query, top_k=3):
    """
    Find the most relevant knowledge-base chunks for a user question.
    """

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.array(query_embedding, dtype="float32")

    # Search FAISS
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "source": df_chunks.iloc[idx]["source"],
            "category": df_chunks.iloc[idx]["category"],
            "content": df_chunks.iloc[idx]["content"]
        })

    return results


print("Retrieval function created successfully!")

Retrieval function created successfully!


In [8]:
query = "How long is the warranty for the CoolMax 300 refrigerator?"

results = retrieve_chunks(query, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(result["content"])


--- Result 1 ---
Score: 0.5180
Source: refrigerators.txt
HOMEHUB APPLIANCES
PRODUCT CATALOG — REFRIGERATORS

Product: HomeHub CoolMax 300
Model: HCR-300X
Category: Refrigerator

Capacity: 320 liters
Energy Rating: 4 Star
Refrigerator Type: Frost Free
Compressor Type: Inverter
Color: Silver
Annual Energy Consumption: 285 kWh
Price: PKR 184,999

Warranty:
The product includes a 2-year complete appliance warranty and a
10-year compressor warranty, subject to the manufacturer's warranty
terms.

Installation:
Eligible refrigerator purchases include complimentary standard
installation within Karachi.

Delivery:
Standard delivery within Karachi normally takes 2–4 business days.
Delivery outside Karachi may require additional time and charges.

Additional Information:
The refrigerator includes adjustable shelves, a digital temperature
display, a dedicat

--- Result 2 ---
Score: 0.4105
Source: retailer_faq.txt
to eligible products
and standard installation requirements. Additional work may inc

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)
print("API key length:", len(api_key) if api_key else 0)

API key loaded: True
API key length: 53


In [9]:
from google import genai

client = genai.Client(api_key=api_key)

print("Gemini client created successfully!")

Gemini client created successfully!


In [10]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain in one sentence what a refrigerator does."
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ConnectError: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1082)

In [11]:
def build_context(results):
    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"[Source {i}: {result['source']}]\n"
            f"{result['content']}"
        )

    return "\n\n".join(context_parts)


# Test with our warranty question
query = "How long is the warranty for the CoolMax 300 refrigerator?"

results = retrieve_chunks(query, top_k=3)
context = build_context(results)

print(context)

NameError: name 'retrieve_chunks' is not defined

In [13]:
def rag_answer(query, top_k=3):
    # 1. Retrieve relevant chunks
    results = retrieve_chunks(query, top_k=top_k)

    # 2. Build context from retrieved chunks
    context = build_context(results)

    # 3. Create a grounded prompt
    prompt = f"""
You are the customer-support assistant for HomeHub Appliances,
a fictional independent home appliance retailer in Karachi, Pakistan.

Answer the user's question using ONLY the information provided
in the knowledge base context below.

Rules:
- Do not invent information.
- Do not use outside knowledge.
- If the answer is not available in the context, clearly say:
  "I couldn't find that information in the HomeHub knowledge base."
- Be concise and helpful.
- Mention the relevant source document(s) at the end.

KNOWLEDGE BASE CONTEXT:
{context}

USER QUESTION:
{query}
"""

    # 4. Generate the answer using Gemini
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text, results

In [14]:
query = "How long is the warranty for the CoolMax 300 refrigerator?"

answer, retrieved_results = rag_answer(query)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")
for result in retrieved_results:
    print(f"- {result['source']} (score: {result['score']:.4f})")

NameError: name 'retrieve_chunks' is not defined